# CAF Remote Computing tutorial using Globus

This tutorial notebook shows how to run computations and agents on remote HPC systems using Globus Compute and Academy. 

Globus Compute is a managed computing service. In addition to providing a simple Python/REST API for sending tasks to remote computers, it also provides a ``fire-and-forget'' computing model in which a managed cloud service takes responsibility for orchestrating execution of tasks on a remote endpoint. Users can retrieve results after completiion via the cloud service. 

This tutorial is configured to use the Globus Compute endpoint hosted by ALCF on the [*Polaris*](https://www.alcf.anl.gov/polaris) computer.  You can also set up your own endpoint on resources to which you have access by following the [Globus Compute documentation](https://globus-compute.readthedocs.io/en/latest/endpoints/endpoints.html).  Globus Compute endpoints can be deployed on many cloud platforms, clusters with batch schedulers (e.g., Slurm, PBS), Kubernetes, or on a local PC.  After configuring an endpoint you can use it in this tutorial by simply setting the `endpoint_id` below.



### Configuration

We first create a Globus Compute executor that we will use to submit and manage tasks.  The executor provides a simple asynchronous Python interface.  Note: the first time you create the executor you will need to authenticate with Globus Auth. To use the Polaris endpoint you must authenticate using an ALCF identity.  

Globus Compute Multi User Endpoints spawn a unique endpoint process for each user and subsequently
allow task exection. The user endpoint is configured according to the ``user_endpoint_config`` argument.  To use Polaris you must specify: 1) our account and 2) the queue to use. You can optionally specify a ``config_key`` to configure the execution environment. In this case, we will load a shared UV environment with various packages preinstalled. If you want to install additional tools, you will need to create a custom environment.

In [ ]:
from globus_compute_sdk import Executor
from globus_compute_sdk.serialize import ComputeSerializer, CombinedCode

compute_endpoint = '9a947ba5-f537-4681-acf3-cc66485aadec' # Polaris Globus Compute endpoint

POLARIS_CONFIG = {
    'account': 'APSDataAnalysis',
    'queue': 'debug',
    'config_key': 'source /home/yadunand/setup_openmm.sh'
}

serializer = ComputeSerializer(strategy_code=CombinedCode())

gce = Executor(endpoint_id=compute_endpoint, 
               user_endpoint_config=POLARIS_CONFIG, 
               serializer=serializer)

### 1. Running a function

Globus Compute executes Python "functions". These functions may be pure Python code or may wrap binaries or command line invocations. 

In [ ]:
def hello_world(name: str):
    import platform
    return f"Hello {name} from Polaris\n{list(platform.uname())}"

future = gce.submit(hello_world, "ModCon")

# Globus compute implements an asynchronous model. 
# Block and wait for the result
print(future.result())  

You can also run shell functions by passing arguments to the Globus Compute ``ShellFunction``.

In [ ]:
from globus_compute_sdk import ShellFunction

bf = ShellFunction("echo '{message}'")
future = gce.submit(bf, message='Hello World')
shell_result = future.result()  # ShellFunctions return ShellResults
print(shell_result.stdout)

### 2. Policy-based function routing

A common pattern we see across model teams is the need to dynamically choose which remote computations (e.g., simulations) should be run based on a decision policy (e.g., a LLM).

In this example, we have three functions to compute: prime numbers, molecular energy, and molecular dynamics trajectories. We use an LLM to decide which function to run. 

To remove dependencies and expedite the demo we use mock implementations of the energy and trajectory functions.

In [ ]:
from functools import wraps
def gce_wrapper(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        future = gce.submit(func, *args, **kwargs)
        return future.result()
    return wrapper

@gce_wrapper
def compute_primes(n: int = 1000) -> dict:
    """Count prime numbers up to n."""
    def is_prime(x):
        if x < 2:
            return False
        for i in range(2, int(x**0.5) + 1):
            if x % i == 0:
                return False
        return True

    primes = [x for x in range(2, n + 1) if is_prime(x)]
    return {
        "task": "prime_count",
        "n": n,
        "count": len(primes),
        "largest": primes[-1] if primes else None,
        "first_10": primes[:10]
    }

@gce_wrapper
def compute_energy(molecule: str = "H2O") -> dict:
    """Simulate molecular energy calculation."""
    import random
    # Mock responses - real version would use PySCF or similar
    energies = {"H2O": -76.4, "CO2": -188.5, "CH4": -40.5,"N2": -109.5}
    base_energy = energies.get(molecule, -50.0) + random.uniform(-0.1, 0.1)
    return {
        "task": "energy_calculation",
        "molecule": molecule,
        "energy_hartree": base_energy,
        "method": "surrogate_dft",
        "basis": "cc-pVDZ"
    }

@gce_wrapper
def compute_trajectory(n_atoms: int = 27, n_steps: int = 100) -> dict:
    """Simulate MD trajectory."""
    import random
    temperatures = [300 + random.uniform(-20, 20) for _ in range(n_steps)]
    return {
        "task": "md_simulation",
        "n_atoms": n_atoms,
        "n_steps": n_steps,
        "avg_temperature": sum(temperatures) / len(temperatures),
        "final_temperature": temperatures[-1],
    }


We now implement the decision policy using an OpenAI compatible LLM API. 

Follow these instructions to obtain an API key for the ALCF inference service. Or insert a key, model name, and URL for another OpenAI compatible service. 

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
import json
import random

requests=["How many prime numbers are there below 500?",
        "What is the potential energy of a water molecule",
        "What is the MD trajectory of water molecules"]

api_key= ''
model_name = 'openai/gpt-oss-120b' 
base_url = 'https://inference-api.alcf.anl.gov/resource_server/sophia/vllm/v1'
        
llm = ChatOpenAI(
    model=model_name,
    api_key=api_key,
    base_url=base_url,
)

CHOOSER_PROMPT = """You are a task router. Given a user request, choose which computation to run.

Available computations:
1. "primes" - Count prime numbers (for math/number questions)
2. "energy" - Calculate molecular energy (for chemistry/molecule questions)
3. "trajectory" - Run MD simulation (for physics/dynamics/trajectory questions)
"""

tools = [compute_primes, compute_energy, compute_trajectory]
agent = create_agent(
    llm, 
    tools=tools,
    system_prompt=CHOOSER_PROMPT,
)

# Run the agent
request=random.choice(requests)
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": request}]},
    stream_mode='updates',
    version="v2"
):
    if chunk["type"] == "updates":
        for step, data in chunk["data"].items():
            print(f"step: {step}")
            print(f"content: {data['messages'][-1].content_blocks}")